# PyTorch'ta Tensör Boyutlarını Manipüle Etme



In [8]:
import torch

torch.manual_seed(42)  
print("PyTorch sürümü:", torch.__version__)

PyTorch sürümü: 2.14.1+cpu


## Rastgele girdi tensörü

`(32, 10, 64)` = (batch, sequence, embedding)

In [9]:
x = torch.randn(32, 10, 64)
print("x.shape:", x.shape)

x.shape: torch.Size([32, 10, 64])


## view ile (320, 64) şekline getirme


In [10]:
x_2d = x.view(320, 64)          
print("x_2d.shape:", x_2d.shape)

print("Aynı bellek mi?", x_2d.data_ptr() == x.data_ptr())

x_2d.shape: torch.Size([320, 64])
Aynı bellek mi? True


## Ağırlık matrisi (64, 128) ve `@` ile çarpım

`(320, 64) @ (64, 128) -> (320, 128)`

In [11]:
weight = torch.randn(64, 128)
print("weight.shape:", weight.shape)

out_2d = x_2d @ weight
print("out_2d.shape:", out_2d.shape)

weight.shape: torch.Size([64, 128])
out_2d.shape: torch.Size([320, 128])


## Sonucu (32, 10, 128) şekline geri döndürme

In [12]:
out = out_2d.view(32, 10, 128)
print("out.shape:", out.shape)

out.shape: torch.Size([32, 10, 128])


### Doğrulama



In [13]:
out_direct = x @ weight
print("out_direct.shape:", out_direct.shape)
print("Sonuçlar aynı mı?", torch.allclose(out, out_direct, atol=1e-5))

out_direct.shape: torch.Size([32, 10, 128])
Sonuçlar aynı mı? True


---
## `view` ve `reshape` neden ikisi de var? (contiguous kavramı)

**Bellekte tensörler tek boyutlu bir dizi olarak saklanır.** `(32, 10, 64)` gibi çok boyutlu görünüm, sadece `stride` (adım) bilgisiyle bu dizinin üzerine bindirilmiş bir "bakış açısıdır".
Bir tensör **contiguous** (bitişik) ise elemanların bellekteki sırası, mantıksal (satır-satır, C tipi) sırasıyla aynıdır. `x` için stride `(640, 64, 1)` olur.
`transpose`, `permute` ya da adımlı dilimleme (`x[:, ::2]`) gibi işlemler veriyi taşımaz, sadece stride'ları değiştirir; sonuç **contiguous olmayabilir**.

| | `view` | `reshape` |
|---|---|---|
| Kopya | Asla kopyalamaz, her zaman aynı belleği paylaşır | Mümkünse `view` döndürür, değilse **kopyalar** |
| Contiguous olmayan tensör | Stride'lar uyumsuzsa `RuntimeError` verir | Sorunsuz çalışır (gerekirse kopyalar) |
| Ne zaman kullanılır? | Kopya olmadığından ve bellek paylaşımından emin olmak istediğinde | Kopya olup olmaması umurunda olmadığında, "her durumda çalışsın" istendiğinde |

**Neden ikisi birden var?** `view` açık bir garanti sunar (sıfır kopya, paylaşılan bellek, hata verirse bunu bilirsin). `reshape` ise kolaylık sunar (kopya gerekse bile çalışır). Ancak `reshape` sessizce kopya üretebileceği için, sonucu yerinde değiştirdiğinde orijinalin etkilenip etkilenmeyeceği baştan belli olmaz.




`transpose` sonrası tensör contiguous olmadığı için önce `.contiguous()` çağrılır, ardından `view` kullanılır. Aynı iş `.reshape(batch, seq_len, embed_dim)` ile de yapılabilir; fark, kopyanın açıkça mı yoksa otomatik mi yapıldığıdır.